<a href="https://colab.research.google.com/github/jhonyjm4/SERS_GS1/blob/main/SERS_GS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ollama==0.6.2

import json
from ollama import Client

In [ ]:
# ==== CONFIGURAÇÃO DA IA OPERACIONAL (OLLAMA CLOUD) ====
API_KEY = "3a7a5e76c2914495834ec14ab91a4730.ECpPqDEws6dXZ5ZKi9942xUA"
MODEL_NAME = "gpt-oss:120b"

# Configuração oficial do Cliente Ollama utilizando a sua chave de acesso
client = Client(
    host="https://ollama.com",
    headers={'Authorization': f'Bearer {API_KEY}'}
)

def chamar_ia_controle_missao(prompt_sistema, prompt_usuario):
    """
    Função inteligente de telemetria baseada no Ollama Cloud.
    Conecta ao modelo gpt-oss:120b para tomada de decisão básica e respostas automatizadas.
    """
    # Combinamos o prompt de sistema e de usuário no formato de mensagens esperado pelo Ollama
    mensagens = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": prompt_usuario}
    ]

    try:
        response = client.chat(
            model=MODEL_NAME,
            messages=mensagens,
            options={
                "num_predict": 800, # Limite de tokens para respostas diretas
                "temperature": 0.3   # Baixa temperatura para manter o rigor técnico operacional
            },
            stream=False
        )
        return response['message']['content'].strip()
    except Exception as e:
        # Mecanismo de defesa: se a API falhar ou der timeout, aciona a lógica local
        return f"[Alerta de Conexão Ollama: {e} - Acionando Inteligência de Contingência Local]"


# ==== DADOS OPERACIONAIS DA MISSÃO ====
nome_missao = "Mission Orion"
nome_equipe = "Equipe Apollo"
lista_pontos_missao = []

# ==== VALORES MÁXIMOS POR FAIXA ====
temperatura_critica_maxima = 35
temperatura_critica_minima = 18
temperatura_atencao = 33

comunicacao_critica = 30
comunicacao_atencao = 65

energia_critica = 20
energia_atencao = 50

suporte_vida_critico = 80
suporte_vida_atencao = 90

status_modulos_critica = 40
status_modulos_atencao = 65

dados_missao = [
    [28.3, 90, 95, 98.2, 95],
    [30.7, 85, 90, 95.5, 90],
    [31.2, 25, 85, 92.4, 88],
    [33.4, 40, 18, 90.1, 80],
    [35.6, 20, 15, 75.6, 35],
    [32.0, 70, 45, 88.2, 75]
]

numero_ciclos = len(dados_missao)

areas_monitoradas = [
    "Temperatura interna",
    "Comunicação com a base",
    "Sistemas de energia",
    "Suporte de vida",
    "Status dos módulos"
]

ciclos_monitorados = [
    "início da missão",
    "estabilização dos sistemas",
    "queda parcial de comunicação",
    "alerta de energia",
    "risco operacional",
    "tentativa de recuperação"
]

# ==== FUNÇÕES DE ANÁLISE ====
def analisar_temperatura(ciclo):
    if dados_missao[ciclo][0] >= temperatura_critica_maxima or dados_missao[ciclo][0] <= temperatura_critica_minima:
        return 2
    elif dados_missao[ciclo][0] < temperatura_critica_maxima and dados_missao[ciclo][0] >= temperatura_atencao:
        return 1
    else:
        return 0

def analisar_comunicacao(ciclo):
    if dados_missao[ciclo][1] <= comunicacao_critica:
        return 2
    elif dados_missao[ciclo][1] > comunicacao_critica and dados_missao[ciclo][1] <= comunicacao_atencao:
        return 1
    else:
        return 0

def analisar_energia(ciclo):
    if dados_missao[ciclo][2] <= energia_critica:
        return 2
    elif dados_missao[ciclo][2] > energia_critica and dados_missao[ciclo][2] <= energia_atencao:
        return 1
    else:
        return 0

def analisar_suporte_vida(ciclo):
    if dados_missao[ciclo][3] <= suporte_vida_critico:
        return 2
    elif dados_missao[ciclo][3] > suporte_vida_critico and dados_missao[ciclo][3] <= suporte_vida_atencao:
        return 1
    else:
        return 0

def analisar_status_modulos(ciclo):
    if dados_missao[ciclo][4] <= status_modulos_critica:
        return 2
    elif dados_missao[ciclo][4] > status_modulos_critica and dados_missao[ciclo][4] <= status_modulos_atencao:
        return 1
    else:
        return 0

# ==== FUNÇÕES DE CÁLCULO ====
def somar_pontos_risco_ciclo(ciclo):
    pontos_risco = sum([
        analisar_temperatura(ciclo),
        analisar_comunicacao(ciclo),
        analisar_energia(ciclo),
        analisar_suporte_vida(ciclo),
        analisar_status_modulos(ciclo)
    ])
    return pontos_risco

def calc_media_area_missao(info):
    soma_info_missao = 0
    for i in range(numero_ciclos):
        soma_info_missao += dados_missao[i][info]
    return soma_info_missao / numero_ciclos

def calc_media_risco_missao():
    soma_risco_missao = 0
    for i in range(numero_ciclos):
        soma_risco_missao += somar_pontos_risco_ciclo(i)
    return soma_risco_missao / numero_ciclos

# ==== FUNÇÕES AUXILIARES ====
def pontuacao_para_status(pontos):
    match pontos:
        case 0:
            return "NORMAL"
        case 1:
            return "ATENÇÃO"
        case 2:
            return "CRÍTICO"

def listar_pontos_areas(ciclo):
    pontos_area = []
    pontos_area.append(analisar_temperatura(ciclo))
    pontos_area.append(analisar_comunicacao(ciclo))
    pontos_area.append(analisar_energia(ciclo))
    pontos_area.append(analisar_suporte_vida(ciclo))
    pontos_area.append(analisar_status_modulos(ciclo))
    return pontos_area

def classificar_ciclo(ciclo):
    pontos_risco = somar_pontos_risco_ciclo(ciclo)
    if 0 <= pontos_risco <= 2:
        return "MISSÃO ESTÁVEL"
    elif 3 <= pontos_risco <= 5:
        return "MISSÃO EM ATENÇÃO"
    else:
        return "MISSÃO CRÍTICA"

def classificar_missao(risco_medio):
    if risco_medio >= 6:
        return "MISSÃO EM ESTADO CRÍTICO"
    elif risco_medio >= 3 and risco_medio <= 5:
        return "MISSÃO EM ATENÇÃO"
    else:
        return "MISSÃO ESTÁVEL"

def identificar_area_mais_afetada_ciclo(ciclo):
    pontos_area = listar_pontos_areas(ciclo)
    maior_pontuacao = max(pontos_area)

    if maior_pontuacao == 0:
        return "Nenhuma área com riscos"

    indices_areas_mais_afetadas = [i for i, x in enumerate(pontos_area) if x == maior_pontuacao]
    areas = []
    for indice in indices_areas_mais_afetadas:
        areas.append(areas_monitoradas[indice])

    if len(areas) == 1:
        return areas[0]
    else:
        return ", ".join(areas[:-1]) + " e " + areas[-1]

def identificar_ciclo_mais_critico():
    lista_soma_pontos_ciclo = []
    for i in range(numero_ciclos):
        lista_soma_pontos_ciclo.append(somar_pontos_risco_ciclo(i))

    maior_risco_missao = max(lista_soma_pontos_ciclo)
    ciclos_criticos = []
    indices_ciclos_mais_criticos = [i for i, x in enumerate(lista_soma_pontos_ciclo) if x == maior_risco_missao]

    for indice in indices_ciclos_mais_criticos:
        ciclos_criticos.append(str(indice+1))

    if len(ciclos_criticos) == 1:
        return ciclos_criticos[0]
    else:
        return ", ".join(ciclos_criticos[:-1]) + " e " + ciclos_criticos[-1]

def numero_ciclos_criticos():
    quantidade_ciclos_criticos = 0
    for i in range(numero_ciclos):
        if classificar_ciclo(i) == "MISSÃO CRÍTICA":
            quantidade_ciclos_criticos += 1
    return quantidade_ciclos_criticos

def analisar_tendencia_ciclo(ciclo):
    if ciclo == 0:
        return "Sem histórico"

    risco_atual = somar_pontos_risco_ciclo(ciclo)
    risco_anterior = somar_pontos_risco_ciclo(ciclo - 1)

    if risco_atual == risco_anterior:
        return "Estável"
    elif risco_atual > risco_anterior:
        return "Piora"
    else:
        return "Melhora"

# ==== FUNÇÕES GENERATIVAS INTEGRADAS COM OLLAMA ====
def gerar_recomendacao(ciclo):
    """
    Utiliza o modelo gpt-oss:120b do Ollama Cloud para gerar diretrizes dinâmicas de engenharia.
    """
    recomendacoes = []
    if analisar_temperatura(ciclo) == 2:
        recomendacoes.append("verificar controle térmico")
    if analisar_comunicacao(ciclo) == 2:
        recomendacoes.append("restabelecer contato com a base")
    if analisar_energia(ciclo) == 2:
        recomendacoes.append("ativar modo de economia de energia")
    if analisar_suporte_vida(ciclo) == 2:
        recomendacoes.append("acionar protocolo de suporte à vida")
    if analisar_status_modulos(ciclo) == 2:
        recomendacoes.append("reduzir operações não essenciais")

    if not recomendacoes:
        texto_contingencia = "Nenhuma ação necessária"
    else:
        if len(recomendacoes) == 1:
            texto_contingencia = recomendacoes[0]
        else:
            texto_contingencia = ", ".join(recomendacoes[:-1]) + " e " + recomendacoes[-1]

    # Economia de requisições à nuvem se o status do ciclo for 100% nominal
    if somar_pontos_risco_ciclo(ciclo) == 0:
        return "Sistemas nominais. Nenhuma intervenção de emergência requerida."

    prompt_sistema = (
        "Você é a Inteligência Artificial embarcada na nave Orion. "
        "Sua função é ler os dados críticos de telemetria e gerar um comando de engenharia direto, "
        "curto (no máximo duas frases) e estritamente técnico para a tripulação."
    )

    prompt_usuario = (
        f"Alerta operacional no Ciclo {ciclo + 1}: {ciclos_monitorados[ciclo].upper()}.\n"
        f"Telemetria recebida:\n"
        f"- Temperatura Interna: {dados_missao[ciclo][0]} °C\n"
        f"- Link de Comunicação: {dados_missao[ciclo][1]}%\n"
        f"- Sistemas de Energia: {dados_missao[ciclo][2]}%\n"
        f"- Suporte de Vida: {dados_missao[ciclo][3]}%\n"
        f"- Integridade dos Módulos: {dados_missao[ciclo][4]}%\n"
        f"Diretriz básica de segurança local: {texto_contingencia}.\n"
        f"Gere a ordem técnica acionável."
    )

    resposta_ia = chamar_ia_controle_missao(prompt_sistema, prompt_usuario)

    # Se retornar mensagem de erro/alerta do bloco try/except, devolve a contingência limpa
    if "[Alerta" in resposta_ia:
        return f"{texto_contingencia} (Modo de Contingência Local Ativo)"

    return resposta_ia


def gerar_relatorio_final():
    soma_temperatura = 0
    soma_comunicacao = 0
    soma_energia_sistemas = 0
    soma_suporte_vida = 0
    soma_status_modulos = 0

    for i in range(numero_ciclos):
        pontos_area = listar_pontos_areas(i)
        soma_temperatura += pontos_area[0]
        soma_comunicacao += pontos_area[1]
        soma_energia_sistemas += pontos_area[2]
        soma_suporte_vida += pontos_area[3]
        soma_status_modulos += pontos_area[4]

    risco_inicial_missao = somar_pontos_risco_ciclo(0)
    risco_final_missao = somar_pontos_risco_ciclo(numero_ciclos-1)

    if risco_final_missao == risco_inicial_missao:
        tendencia = "Estável"
    elif risco_final_missao > risco_inicial_missao:
        tendencia = "Piora"
    else:
        tendencia = "Melhora"

    pontuacoes = [
        soma_temperatura,
        soma_comunicacao,
        soma_energia_sistemas,
        soma_suporte_vida,
        soma_status_modulos
    ]

    maior_pontuacao = max(pontuacoes)
    indices_areas_mais_afetadas = [i for i, x in enumerate(pontuacoes) if x == maior_pontuacao]
    areas_mais_afetadas_missao = [areas_monitoradas[indice] for indice in indices_areas_mais_afetadas]

    print("============================================== RELATÓRIO FINAL ==============================================")
    print("")
    print(f"Missão: {nome_missao}")
    print(f"Equipe: {nome_equipe}")
    print(f"Quantidade de ciclos analisados: {numero_ciclos}")
    print("")

    if len(areas_mais_afetadas_missao) == 1:
        print(f"A área mais afetada da missão foi {areas_mais_afetadas_missao[0]} com {maior_pontuacao} pontos")
    else:
        texto_areas = (", ".join(areas_mais_afetadas_missao[:-1]) + " e " + areas_mais_afetadas_missao[-1])
        print(f"As áreas mais afetadas da missão foram: {texto_areas} com {maior_pontuacao} pontos")

    print("")
    print(f"- Tendência geral da missão: {tendencia}")
    print(f"- Risco do primeiro ciclo da missão: {risco_inicial_missao} pontos")
    print(f"- Risco do último ciclo da missão: {risco_final_missao} pontos")
    print("")

    print("-- Pontuação acumulada por área --")
    print("")
    print(f"- Temperatura interna: {soma_temperatura} pontos de risco na missão")
    print(f"- Comunicação com a base: {soma_comunicacao} pontos de risco na missão")
    print(f"- Sistemas de energia: {soma_energia_sistemas} pontos de risco na missão")
    print(f"- Suporte de vida: {soma_suporte_vida} pontos de risco na missão")
    print(f"- Status dos módulos: {soma_status_modulos} pontos de risco na missão")
    print("")
    print(f"- Ciclo mais crítico: Ciclo {identificar_ciclo_mais_critico()}")

    lista_soma_pontos_ciclo = []
    for i in range(numero_ciclos):
        lista_soma_pontos_ciclo.append(somar_pontos_risco_ciclo(i))

    maior_risco_missao = max(lista_soma_pontos_ciclo)

    print(f"- Maior pontuação de risco: {maior_risco_missao}")
    print(f"- Risco médio da missão: {calc_media_risco_missao()}")
    print(f"- Quantidade de ciclos críticos: {numero_ciclos_criticos()}")
    print("")

    print("-- Média de dados de cada área --")
    print("")
    print(f"- Média de temperatura: {calc_media_area_missao(0):.2f} ºC")
    print(f"- Média de comunicação: {calc_media_area_missao(1):.2f}%")
    print(f"- Média de energia: {calc_media_area_missao(2):.2f}%")
    print(f"- Média de suporte de vida: {calc_media_area_missao(3):.2f}%")
    print(f"- Média de status dos módulos: {calc_media_area_missao(4):.2f}%")

    print("")
    print(f"Classificação final da missão: {classificar_missao(calc_media_risco_missao())}")
    print("")

    print("CONCLUSÃO LOCAL:")
    print("A missão Mission Orion apresentou instabilidades ao longo dos ciclos monitorados, com seu momento mais crítico ocorrendo no Ciclo 5, ")
    print("que atingiu 10 pontos de risco. As áreas mais afetadas foram Comunicação com a Base e Sistemas de Energia, ambas com 5 pontos acumulados. ")
    print("Apesar da tendência geral de piora da missão, observou-se uma recuperação parcial no último ciclo, reduzindo o risco para 2 pontos. Com risco ")
    print("médio de 3,0 pontos e classificação final de MISSÃO EM ATENÇÃO, recomenda-se manter o monitoramento dos sistemas de comunicação e energia para ")
    print("prevenir novas ocorrências críticas.")
    print("-" * 110)

    # ==== PARECER DA IA VOLTADO A SUSTENTABILIDADE (REQUISITO DA GLOBAL SOLUTION) ====
    print("PARECER COGNITIVO DA IA (ANÁLISE DE SUSTENTABILIDADE ENERGÉTICA ESPACIAL):")

    prompt_sistema_relatorio = (
        "Você é o Diretor de Sustentabilidade e Infraestrutura de Energia da Agência Apollo. "
        "Analise o histórico completo de telemetria da missão e elabore um parecer técnico descritivo de até 4 linhas "
        "propondo melhorias em eficiência energética e uso de recursos renováveis para as próximas fases."
    )

    prompt_usuario_relatorio = (
        f"Métricas Consolidadas da Missão Orion:\n"
        f"- Classificação de Operação: {classificar_missao(calc_media_risco_missao())}\n"
        f"- Áreas de Maior Desgaste Acumulado: {', '.join(areas_mais_afetadas_missao)}\n"
        f"- Eficiência Média da Energia: {calc_media_area_missao(2):.2f}%\n"
        f"- Estabilidade Média dos Módulos: {calc_media_area_missao(4):.2f}%\n"
        f"Gere uma análise preditiva e focada em sustentabilidade de hardware."
    )

    conclusao_ia = chamar_ia_controle_missao(prompt_sistema_relatorio, prompt_usuario_relatorio)
    print(conclusao_ia)
    print("==================================================================================================================================================")


# ==== REGISTRO DE CICLOS ====
print("=======================================================================================================================")
print("MISSION CONTROL AI")
print("=======================================================================================================================")
print("")
print(f"Missão: {nome_missao}")
print(f"Equipe: {nome_equipe}")
print(f"Quantidade de ciclos analisados: {numero_ciclos}")
print("")

for i in range(numero_ciclos):
    lista_pontos_missao.append(somar_pontos_risco_ciclo(i))

    print(f"============================================== CICLO {i+1}: {ciclos_monitorados[i].upper()} ==============================================")
    print("")

    print(f"- Temperatura interna: {dados_missao[i][0]} °C | {pontuacao_para_status(analisar_temperatura(i))}")
    print(f"- Comunicação com a base: {dados_missao[i][1]} % | {pontuacao_para_status(analisar_comunicacao(i))}")
    print(f"- Sistemas de energia: {dados_missao[i][2]} % | {pontuacao_para_status(analisar_energia(i))}")
    print(f"- Suporte de vida: {dados_missao[i][3]} % | {pontuacao_para_status(analisar_suporte_vida(i))}")
    print(f"- Status dos módulos: {dados_missao[i][4]} % | {pontuacao_para_status(analisar_status_modulos(i))}")

    print("")
    print(f"- Análise de tendência entre esse ciclo e o último: {analisar_tendencia_ciclo(i)}")
    print("")
    print(f"- Pontuação de risco do ciclo: {somar_pontos_risco_ciclo(i)}")
    print(f"- Classificação do ciclo: {classificar_ciclo(i)}")
    print(f"- Área(s) de maior risco: {identificar_area_mais_afetada_ciclo(i)}")
    print("")
    print(f"- Recomendações da IA (Ollama): {gerar_recomendacao(i)}")
    print("")

# ==== EXECUÇÃO DO RELATÓRIO FINAL ====
gerar_relatorio_final()

MISSION CONTROL AI

Missão: Mission Orion
Equipe: Equipe Apollo
Quantidade de ciclos analisados: 6

============================================== CICLO 1: INÍCIO DA MISSÃO ==============================================

- Temperatura interna: 28.3 °C | NORMAL
- Comunicação com a base: 90 % | NORMAL
- Sistemas de energia: 95 % | NORMAL
- Suporte de vida: 98.2 % | NORMAL
- Status dos módulos: 95 % | NORMAL

- Análise de tendência entre esse ciclo e o último: Sem histórico

- Pontuação de risco do ciclo: 0
- Classificação do ciclo: MISSÃO ESTÁVEL
- Área(s) de maior risco: Nenhuma área com riscos

- Recomendações da IA (Ollama): Sistemas nominais. Nenhuma intervenção de emergência requerida.

============================================== CICLO 2: ESTABILIZAÇÃO DOS SISTEMAS ==============================================

- Temperatura interna: 30.7 °C | NORMAL
- Comunicação com a base: 85 % | NORMAL
- Sistemas de energia: 90 % | NORMAL
- Suporte de vida: 95.5 % | NORMAL
- Status dos módul